# Miniforge — Real Benchmark Notebook

**MiniMax M2.7 GGUF inference library benchmarks — all inference is real.**

This notebook:
1. Mounts Google Drive and installs all dependencies
2. Runs the full pytest test suite
3. Loads a real GGUF model via the Miniforge library
4. Runs real throughput, memory, quality, and context benchmarks
5. Generates charts from actual measurements
6. Saves all results + figures to Drive

> **No simulated data.** Every number comes from actual model inference.

## 0  Configuration

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
# MiniMax M2.7 (228B MoE) — the actual model this library targets.
# UD-Q4_K_M ≈ 128 GB on disk, well under the 230 GB cap.
# Other options (comment/uncomment):
#   UD-IQ2_XXS  ≈  61 GB  (smallest, lowest quality)
#   UD-IQ3_XXS  ≈  75 GB
#   UD-Q4_K_M   ≈ 128 GB  ← default
#   UD-Q5_K_M   ≈ 160 GB
#   UD-Q6_K     ≈ 192 GB
#   UD-Q8_K_XL  ≈ 228 GB  (best integer quant, right at the cap)
MODEL_ID     = 'unsloth/MiniMax-M2.7-GGUF'
QUANTIZATION = 'UD-Q4_K_M'

# Model is downloaded to Colab local SSD (/content/) — NOT Google Drive.
# Results, figures, and the JSON report go to Drive.
OUTPUT_DIR = '/content/drive/MyDrive/miniforge_results'

# Benchmark settings for Colab Pro
BENCH_WARMUP     = 2
BENCH_ITERATIONS = 5
MAX_TOKENS       = 512
MAX_CONTEXT      = 16384

print(f'Model : {MODEL_ID}  ({QUANTIZATION})')
print(f'Output: {OUTPUT_DIR}')

## 1  Mount Google Drive

In [ ]:
import os, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    OUTPUT_DIR = './miniforge_results'

# Results go to Drive; model goes to local /content/ SSD (not Drive)
os.makedirs(OUTPUT_DIR, exist_ok=True)
MODEL_CACHE = '/content/models'   # local SSD only — not synced to Drive
os.makedirs(MODEL_CACHE, exist_ok=True)

# Show available disk space on /content/
df = subprocess.run(['df', '-h', '/content'], capture_output=True, text=True)
print(df.stdout)
print(f'Results → Drive  : {OUTPUT_DIR}')
print(f'Model   → Local  : {MODEL_CACHE}  (not Google Drive)')

## 2  Install Dependencies

In [ ]:
import subprocess, sys

# Detect GPU
has_cuda = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
if has_cuda:
    gpu_info = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    print(f'GPU: {gpu_info.stdout.strip()}')
else:
    print('No GPU detected — CPU only')

# Clone / update miniforge repo
if IN_COLAB:
    !git clone https://github.com/Jackson57279/miniforge.git /content/miniforge 2>/dev/null || \
        (cd /content/miniforge && git pull)
    REPO_ROOT = '/content/miniforge'
else:
    REPO_ROOT = os.path.abspath('.')

# Core Python deps
!pip install -q \
    transformers>=4.40.0 \
    psutil>=5.9 \
    pydantic>=2.0 \
    pyyaml>=6.0 \
    tqdm>=4.66 \
    numpy>=1.24 \
    matplotlib>=3.8 \
    seaborn>=0.13 \
    plotly>=5.20 \
    kaleido \
    huggingface_hub>=0.20 \
    nest_asyncio \
    pytest pytest-asyncio

# llama-cpp-python — GPU build for Colab Pro, CPU fallback
if has_cuda:
    !pip install -q llama-cpp-python \
        --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
        || CMAKE_ARGS="-DGGML_CUDA=on" pip install -q llama-cpp-python
else:
    !pip install -q llama-cpp-python \
        --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
        || pip install -q llama-cpp-python

# Install miniforge itself
!pip install -q -e {REPO_ROOT} --no-deps

print('\nInstallation complete.')

## 3  Imports & Helpers

In [ ]:
import sys, asyncio, json, time, gc, threading
import psutil
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime
import nest_asyncio

# Allow asyncio.run() inside an already-running loop (Colab)
nest_asyncio.apply()

sys.path.insert(0, f'{REPO_ROOT}/src')
sys.path.insert(0, REPO_ROOT)

# ── Style ────────────────────────────────────────────────────────────────────
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.15)
PALETTE = sns.color_palette('muted')
matplotlib.rcParams.update({'figure.dpi': 150, 'savefig.bbox': 'tight'})
TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

def save(fig, name: str):
    path = f'{OUTPUT_DIR}/{name}_{TIMESTAMP}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  Saved → {path}')
    plt.show()

def save_plotly(fig, name: str):
    png_path  = f'{OUTPUT_DIR}/{name}_{TIMESTAMP}.png'
    html_path = f'{OUTPUT_DIR}/{name}_{TIMESTAMP}.html'
    try:
        fig.write_image(png_path)
    except Exception:
        pass  # kaleido optional
    fig.write_html(html_path)
    print(f'  Saved → {html_path}')
    fig.show()

print('Imports OK.')

## 4  Unit Tests (real pytest run)

In [ ]:
!cd {REPO_ROOT} && python -m pytest tests/ -v --tb=short 2>&1 | tee /tmp/pytest_output.txt

with open('/tmp/pytest_output.txt') as f:
    pytest_output = f.read()

passed  = pytest_output.count(' PASSED')
failed  = pytest_output.count(' FAILED')
errors  = pytest_output.count(' ERROR')
skipped = pytest_output.count(' SKIPPED')
print(f'\nTest summary: {passed} passed, {failed} failed, {errors} errors, {skipped} skipped')

In [ ]:
# Figure 1: Test results donut
fig, ax = plt.subplots(figsize=(6, 6))

labels, sizes, colors = [], [], []
if passed:  labels.append('Passed');  sizes.append(passed);  colors.append('#4caf50')
if failed:  labels.append('Failed');  sizes.append(failed);  colors.append('#f44336')
if errors:  labels.append('Errors');  sizes.append(errors);  colors.append('#ff9800')
if skipped: labels.append('Skipped'); sizes.append(skipped); colors.append('#9e9e9e')

if not sizes:
    labels, sizes, colors = ['No tests found'], [1], ['#9e9e9e']

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors,
    autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(width=0.55)
)
for at in autotexts:
    at.set_fontsize(13)

total = sum(sizes)
ax.text(0, 0, f'{total}\ntotal', ha='center', va='center', fontsize=14, fontweight='bold')
ax.set_title('Unit Test Results (real pytest run)', fontsize=16, fontweight='bold', pad=20)

save(fig, 'fig01_test_results')

## 5  Load Model

In [ ]:
from miniforge.models.minimax import Miniforge
from miniforge.utils.config import M7Config
from miniforge.core.memory import MemoryManager
import psutil, subprocess

# ── Detect GPU ────────────────────────────────────────────────────────────────
has_cuda = subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
if has_cuda:
    vram_result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True
    )
    lines = vram_result.stdout.strip().split('\n')
    gpu_name = lines[0].split(',')[0].strip() if lines else 'unknown'
    try:
        vram_mb = int(lines[0].split(',')[1].strip())
    except Exception:
        vram_mb = 40000
    n_gpu_layers = -1
    print(f'GPU : {gpu_name}  ({vram_mb / 1024:.1f} GB VRAM) → full offload')
else:
    vram_mb = 0
    n_gpu_layers = 0
    print('No GPU detected — CPU inference with mmap expert paging')

# ── System RAM ────────────────────────────────────────────────────────────────
n_cpus       = psutil.cpu_count(logical=False) or 2
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f'RAM : {total_ram_gb:.1f} GB   CPUs: {n_cpus}')

# ── Config ────────────────────────────────────────────────────────────────────
config = M7Config(
    n_threads    = n_cpus,
    max_memory_gb= total_ram_gb * 0.85,
    n_ctx        = MAX_CONTEXT,
    n_gpu_layers = n_gpu_layers,
    flash_attn   = has_cuda,
    use_mmap     = True,   # mmap experts from /content/ SSD
)

print(f'\nConfig: n_ctx={config.n_ctx}, n_gpu_layers={config.n_gpu_layers}, '
      f'flash_attn={config.flash_attn}')

# ── Download & load via Miniforge ─────────────────────────────────────────────
# download_dir points to local /content/models — never touches Google Drive
print(f'\nDownloading {MODEL_ID} [{QUANTIZATION}] to {MODEL_CACHE} ...')
print('(multi-shard GGUF — this will take a while on first run)\n')

load_start = time.perf_counter()
model = await Miniforge.from_pretrained(
    model_id     = MODEL_ID,
    quantization = QUANTIZATION,
    config       = config,
    download_dir = MODEL_CACHE,
)
load_time = time.perf_counter() - load_start

print(f'\nModel ready in {load_time:.1f}s')
print('Memory stats:', model.get_memory_stats())

## 6  Memory Manager Analysis

In [ ]:
from miniforge.core.memory import MemoryManager

mem = model._memory_manager
stats = mem.get_stats()

print(f'System RAM      : {stats.total_gb:.1f} GB')
print(f'Available       : {stats.available_gb:.1f} GB')
print(f'Used            : {stats.used_gb:.1f} GB ({stats.percent_used:.1f}%)')
print(f'Max usable      : {mem.max_usable_gb:.1f} GB')

In [ ]:
# Figure 2: RAM breakdown
live_stats   = mem.get_stats()
proc_rss_gb  = psutil.Process().memory_info().rss / (1024 ** 3)
os_reserve   = max(live_stats.total_gb * 0.10, 1.5)
model_mem    = proc_rss_gb
free_mem     = max(live_stats.available_gb - 0.5, 0)
other_mem    = max(live_stats.total_gb - os_reserve - model_mem - free_mem, 0)

fig, ax = plt.subplots(figsize=(9, 4))
categories = ['System RAM']
bottoms = [0]
for val, label, color in [
    (os_reserve,  f'OS Reserve ({os_reserve:.1f} GB)',      '#ef5350'),
    (model_mem,   f'This Process ({model_mem:.1f} GB)',     '#42a5f5'),
    (other_mem,   f'Other Processes ({other_mem:.1f} GB)',  '#ffa726'),
    (free_mem,    f'Free ({free_mem:.1f} GB)',              '#bdbdbd'),
]:
    ax.barh(categories, [val], left=bottoms, color=color, label=label, height=0.5)
    bottoms = [bottoms[0] + val]

ax.set_xlabel('Memory (GB)', fontsize=12)
ax.set_title(f'Live RAM Allocation — {live_stats.total_gb:.1f} GB Total', fontsize=14, fontweight='bold')
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.50), ncol=2)
ax.set_xlim(0, live_stats.total_gb * 1.05)

save(fig, 'fig02_ram_allocation')

## 7  Quantization Size Chart

In [ ]:
# Figure 3: Model sizes for 228B MoE (real calculations)
model_fp16_gb = 228 * 2
quants = {
    'FP16':      1.000,
    'Q8_0':      0.500,
    'Q6_K':      0.375,
    'Q5_K_M':    0.313,
    'Q4_K_M':    0.250,
    'Q3_K_M':    0.188,
    'IQ2_XXS':   0.134,
    'UD-TQ1_0':  0.083,
}

labels = list(quants.keys())
sizes  = [model_fp16_gb * r for r in quants.values()]
target_gb = mem.max_usable_gb  # actual usable RAM from MemoryManager
bar_colors = ['#4caf50' if s <= target_gb else '#ef5350' for s in sizes]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, sizes, color=bar_colors, edgecolor='white', linewidth=1.2)
ax.axhline(target_gb, color='orange', linestyle='--', linewidth=2,
           label=f'{target_gb:.1f} GB usable RAM (this system)')

for bar, sz in zip(bars, sizes):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{sz:.0f} GB', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Size (GB)', fontsize=12)
ax.set_title('228B MoE Model Size by Quantization', fontsize=14, fontweight='bold')
green_patch = mpatches.Patch(color='#4caf50', label='Fits in RAM')
red_patch   = mpatches.Patch(color='#ef5350', label='Exceeds RAM')
ax.legend(handles=[green_patch, red_patch,
                   plt.Line2D([0],[0], color='orange', linestyle='--', linewidth=2,
                              label=f'{target_gb:.1f} GB budget')])

save(fig, 'fig03_quantization_sizes')

## 8  Context Window vs KV-Cache Memory

In [ ]:
# Figure 4: KV-cache memory vs context window (real MemoryManager calculation)
kv_types = {'f16': 64, 'q8_0': 32, 'q4_0': 16, 'turbo4': 18, 'turbo3': 14}
context_tokens = np.array([1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 194560])

fig, ax = plt.subplots(figsize=(11, 5))
for (kv_name, kb_per_tok), color in zip(kv_types.items(), sns.color_palette('tab10')):
    gb = context_tokens * kb_per_tok * 1024 / (1024 ** 3)
    ax.plot(context_tokens / 1000, gb, marker='o', label=kv_name, color=color, linewidth=2)

budget = mem.max_usable_gb - 2.0  # leave 2 GB for model + overhead
ax.axhline(budget, color='red', linestyle='--', linewidth=2,
           label=f'Usable budget ({budget:.1f} GB)')

ax.set_xlabel('Context Window (K tokens)', fontsize=12)
ax.set_ylabel('KV Cache Memory (GB)', fontsize=12)
ax.set_title('KV Cache Memory vs Context Window by Type', fontsize=14, fontweight='bold')
ax.legend(title='KV type', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.set_xticks(context_tokens / 1000)
ax.set_xticklabels([f'{int(c)}K' if c >= 1 else f'{int(c*1000)}'
                    for c in context_tokens / 1000], rotation=30, ha='right')

save(fig, 'fig04_context_vs_memory')

## 9  Throughput Benchmark (real inference)

In [ ]:
import statistics as _stat

PROMPTS = {
    'code':          'Write a Python function to implement binary search with type hints and docstring.',
    'qa':            'What are the key differences between Python and JavaScript? Compare briefly.',
    'summarization': (
        'Summarize in 3 bullet points: '
        'Artificial intelligence has revolutionized healthcare, transportation, and software. '
        'ML algorithms detect diseases earlier than doctors. AI also optimizes city traffic '
        'flow and creates art. However, AI raises ethical questions about privacy and jobs.'
    ),
    'creative': 'Write a short story opening (2 sentences) about a robot discovering emotions.',
}

throughput_results = {}  # {prompt_type: [tok/s per iteration]}

print('=== Throughput Benchmark ===')
print(f'warmup={BENCH_WARMUP}, iterations={BENCH_ITERATIONS}, max_tokens={MAX_TOKENS}\n')

for ptype, prompt in PROMPTS.items():
    print(f'--- {ptype} ---')
    samples = []

    # Warmup
    for _ in range(BENCH_WARMUP):
        _ = await model.generate(prompt, max_tokens=32)

    # Timed iterations
    for i in range(BENCH_ITERATIONS):
        t0  = time.perf_counter()
        out = await model.generate(prompt, max_tokens=MAX_TOKENS)
        t1  = time.perf_counter()

        elapsed = t1 - t0
        # Word-based token estimate (output only)
        n_tokens = int(len(out.split()) / 0.75)
        tps = n_tokens / elapsed if elapsed > 0 else 0
        samples.append(tps)
        print(f'  iter {i+1}: {tps:.2f} tok/s  ({elapsed*1000:.0f} ms, ~{n_tokens} tokens)')

    throughput_results[ptype] = samples
    mean_tps = _stat.mean(samples)
    print(f'  mean: {mean_tps:.2f} tok/s\n')

print('Throughput benchmark complete.')

In [ ]:
# Figure 6: Throughput boxplot from real data
fig, ax = plt.subplots(figsize=(10, 5))

data_flat  = [v for vals in throughput_results.values() for v in vals]
label_flat = [k for k, vals in throughput_results.items() for _ in vals]

sns.boxplot(x=label_flat, y=data_flat, ax=ax, palette='muted',
            width=0.4, flierprops=dict(marker='o', alpha=0.4))
sns.stripplot(x=label_flat, y=data_flat, ax=ax, color='black',
              size=6, alpha=0.7, jitter=True)

means = {pt: _stat.mean(vals) for pt, vals in throughput_results.items()}
for i, (pt, mean) in enumerate(means.items()):
    ax.text(i, mean + max(data_flat)*0.02, f'{mean:.1f}',
            ha='center', fontsize=10, color='navy', fontweight='bold')

ax.set_xlabel('Prompt Type', fontsize=12)
ax.set_ylabel('Throughput (tokens / second)', fontsize=12)
ax.set_title('Real Inference Throughput by Prompt Type', fontsize=13, fontweight='bold')

save(fig, 'fig06_throughput_boxplot')

## 10  Throughput & TTFT vs Context Length (real)

In [ ]:
# Run at progressively larger context sizes — measures TTFT and generation speed
ctx_sizes = [256, 512, 1024, 2048]
if MAX_CONTEXT >= 4096:
    ctx_sizes.append(4096)

base_text  = 'The quick brown fox jumps over the lazy dog. '
ctx_tps    = []
ctx_ttft   = []

print('=== Context Scaling Benchmark ===\n')
for ctx_len in ctx_sizes:
    # Build a prompt of approximately ctx_len tokens
    target_chars = ctx_len * 4
    prompt = (base_text * ((target_chars // len(base_text)) + 1))[:target_chars]
    prompt += '\n\nSummarize the above in one sentence:'

    # Measure TTFT via streaming
    ttft_samples, tps_samples = [], []

    for _ in range(max(1, BENCH_ITERATIONS - 1)):  # slightly fewer for long contexts
        t_start = time.perf_counter()
        first_token_t = None
        n_out = 0

        stream = await model.generate(prompt, max_tokens=64, stream=True)
        async for tok in stream:
            if first_token_t is None:
                first_token_t = time.perf_counter()
            n_out += 1
        t_end = time.perf_counter()

        if first_token_t:
            ttft_ms = (first_token_t - t_start) * 1000
            gen_sec = t_end - first_token_t
            tps     = (n_out - 1) / gen_sec if gen_sec > 0 else 0
            ttft_samples.append(ttft_ms)
            tps_samples.append(tps)

    mean_ttft = _stat.mean(ttft_samples) if ttft_samples else 0
    mean_tps  = _stat.mean(tps_samples)  if tps_samples  else 0
    ctx_ttft.append(mean_ttft)
    ctx_tps.append(mean_tps)
    print(f'  ctx={ctx_len:6d} → TTFT={mean_ttft:.0f}ms  gen={mean_tps:.2f} tok/s')

print('\nContext benchmark complete.')

In [ ]:
# Figure 7: Throughput vs context (real Plotly chart)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Generation Throughput vs Context', 'Time-to-First-Token vs Context')
)

fig.add_trace(
    go.Scatter(x=ctx_sizes, y=ctx_tps, mode='lines+markers',
               name='tok/s', line=dict(color='royalblue', width=3),
               marker=dict(size=8)),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=ctx_sizes, y=ctx_ttft, mode='lines+markers',
               name='TTFT (ms)', line=dict(color='tomato', width=3),
               marker=dict(size=8)),
    row=1, col=2
)

fig.update_xaxes(title_text='Context Window (tokens)', row=1, col=1)
fig.update_xaxes(title_text='Context Window (tokens)', row=1, col=2)
fig.update_yaxes(title_text='Tokens / second', row=1, col=1)
fig.update_yaxes(title_text='TTFT (ms)', row=1, col=2)

fig.update_layout(
    title='Miniforge Performance vs Context Window (real measurements)',
    height=450, template='plotly_dark', showlegend=True
)

save_plotly(fig, 'fig07_throughput_vs_context')

## 11  Memory Monitoring During Real Inference

In [ ]:
mem_snapshots = []
proc = psutil.Process()

gen_prompt = (
    'Write a detailed 400-word essay explaining the history of artificial intelligence, '
    'covering the 1950s Turing test, 1980s expert systems, 1990s machine learning, '
    'and the current era of large language models.'
)

monitoring = True
sample_times = []

def _mem_sampler():
    while monitoring:
        mem_snapshots.append(proc.memory_info().rss / (1024 ** 2))
        sample_times.append(time.perf_counter())
        time.sleep(0.1)

t = threading.Thread(target=_mem_sampler, daemon=True)
t.start()

print('Generating 512 tokens while sampling memory every 100ms...')
gen_t0 = time.perf_counter()
response = await model.generate(gen_prompt, max_tokens=512)
gen_t1 = time.perf_counter()

monitoring = False
t.join()

gen_duration = gen_t1 - gen_t0
n_words = len(response.split())
gen_tps  = int(n_words / 0.75) / gen_duration

print(f'Generation: {gen_duration:.1f}s  ≈{gen_tps:.1f} tok/s')
print(f'Memory samples collected: {len(mem_snapshots)}')
print(f'Peak RSS: {max(mem_snapshots):.0f} MB')
print(f'Min  RSS: {min(mem_snapshots):.0f} MB')

In [ ]:
# Figure 8: Real memory trace during generation
if sample_times:
    t0_ref = sample_times[0]
    rel_times = [t - t0_ref for t in sample_times]
else:
    rel_times = list(range(len(mem_snapshots)))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(rel_times, mem_snapshots, color='steelblue', linewidth=1.5)
ax.fill_between(rel_times, mem_snapshots, alpha=0.2, color='steelblue')
ax.axhline(np.mean(mem_snapshots), color='red', linestyle='--', alpha=0.8,
           label=f'Mean: {np.mean(mem_snapshots):.0f} MB')
ax.axhline(max(mem_snapshots), color='orange', linestyle=':', alpha=0.8,
           label=f'Peak: {max(mem_snapshots):.0f} MB')

# Mark gen start / end
ax.axvline(0, color='green', linestyle='--', alpha=0.5, label='gen start')
ax.axvline(gen_duration, color='purple', linestyle='--', alpha=0.5, label='gen end')

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Process RSS (MB)', fontsize=12)
ax.set_title('Real Memory Usage During Miniforge Inference (512 tokens)', fontsize=13, fontweight='bold')
ax.legend()

save(fig, 'fig08_memory_during_inference')

## 12  Quality Benchmark (real model evaluation)

In [ ]:
# Real quality benchmark — each question answered by the actual model
import re

QA_PAIRS = [
    {'question': 'What is the capital of France?',
     'expected_keywords': ['paris'],
     'forbidden': ['london', 'berlin', 'rome']},
    {'question': 'Who wrote Romeo and Juliet?',
     'expected_keywords': ['shakespeare'],
     'forbidden': ['hemingway', 'dickens', 'austen']},
    {'question': 'What is 15 multiplied by 7?',
     'expected_keywords': ['105'],
     'forbidden': []},
    {'question': 'What is the largest planet in our solar system?',
     'expected_keywords': ['jupiter'],
     'forbidden': ['saturn', 'earth', 'mars']},
    {'question': 'In which year did World War II end?',
     'expected_keywords': ['1945'],
     'forbidden': ['1939', '1944', '1946']},
]

REASONING_PROBLEMS = [
    {'problem': 'If a train travels 120 km in 2 hours, what is its average speed in km/h?',
     'expected': '60'},
    {'problem': 'A rectangle has length 8 and width 5. What is its area?',
     'expected': '40'},
    {'problem': '3 workers finish a job in 4 days. How many days would 6 workers take?',
     'expected': '2'},
]

INSTRUCTION_TESTS = [
    {'instruction': 'List exactly 3 primary colors separated by commas. Nothing else.',
     'check': lambda r: len([x for x in r.split(',') if x.strip()]) >= 2,
     'desc': 'list with constraints'},
    {'instruction': 'Answer with exactly one word: what is the opposite of hot?',
     'check': lambda r: 'cold' in r.lower() and len(r.split()) <= 3,
     'desc': 'single-word answer'},
    {'instruction': 'Provide a valid JSON object with keys "name" and "age". Only JSON.',
     'check': lambda r: '"name"' in r and '"age"' in r,
     'desc': 'JSON format'},
]

quality_scores = {}

# ── QA ───────────────────────────────────────────────────────────────────────
print('--- QA Accuracy ---')
qa_correct = 0
for item in QA_PAIRS:
    resp = await model.generate(item['question'], max_tokens=50, temperature=0)
    resp_l = resp.lower()
    found   = any(k in resp_l for k in item['expected_keywords'])
    blocked = any(f in resp_l for f in item['forbidden'])
    ok = found and not blocked
    if ok: qa_correct += 1
    print(f'  {"✓" if ok else "✗"} Q: {item["question"][:50]} | got: {resp[:40]}')
qa_acc = qa_correct / len(QA_PAIRS)
quality_scores['QA Accuracy'] = qa_acc
print(f'  Accuracy: {qa_acc:.1%}\n')

# ── Reasoning ────────────────────────────────────────────────────────────────
print('--- Reasoning ---')
reason_correct = 0
for prob in REASONING_PROBLEMS:
    prompt = f"Problem: {prob['problem']}\nSolve step by step, then state the final answer:"
    resp = await model.generate(prompt, max_tokens=200, temperature=0)
    ok = prob['expected'] in resp
    if ok: reason_correct += 1
    print(f'  {"✓" if ok else "✗"} Expected {prob["expected"]} | got: {resp[:60]}')
reason_acc = reason_correct / len(REASONING_PROBLEMS)
quality_scores['Reasoning'] = reason_acc
print(f'  Solve rate: {reason_acc:.1%}\n')

# ── Instruction Following ─────────────────────────────────────────────────────
print('--- Instruction Following ---')
instr_correct = 0
for test in INSTRUCTION_TESTS:
    resp = await model.generate(test['instruction'], max_tokens=80, temperature=0)
    try:
        ok = test['check'](resp)
    except Exception:
        ok = False
    if ok: instr_correct += 1
    print(f'  {"✓" if ok else "✗"} {test["desc"]} | got: {resp[:50]}')
instr_acc = instr_correct / len(INSTRUCTION_TESTS)
quality_scores['Instruction\nFollowing'] = instr_acc
print(f'  Follow rate: {instr_acc:.1%}\n')

# ── Coherence ────────────────────────────────────────────────────────────────
print('--- Coherence ---')
story_prompt = (
    'Write a short story (150-200 words) about a scientist who makes an unexpected discovery. '
    'Include a clear beginning, middle, and end.'
)
story = await model.generate(story_prompt, max_tokens=350, temperature=0.7)
words  = story.split()
unique = set(w.lower() for w in words)
vocab_richness = len(unique) / len(words) if words else 0
has_length = len(words) > 80
coherence_score = min(1.0, vocab_richness * 2 * (1.0 if has_length else 0.5))
quality_scores['Coherence'] = coherence_score
print(f'  Word count: {len(words)}, Vocab richness: {vocab_richness:.2f}')
print(f'  Coherence score: {coherence_score:.2f}')
print(f'  Preview: {story[:120]}...\n')

# ── Summarization ────────────────────────────────────────────────────────────
print('--- Summarization ---')
summ_text = (
    'Artificial intelligence (AI) refers to intelligence demonstrated by machines. '
    'AI research studies intelligent agents that perceive environments and act to maximize goals. '
    'Modern AI includes machine learning, deep learning, and large language models.'
)
summ_prompt = f'Text: {summ_text}\n\nSummarize in one sentence (max 30 words):'
summary = await model.generate(summ_prompt, max_tokens=80, temperature=0)
expected_concepts = ['intelligence', 'machine', 'ai', 'learning', 'agent']
found = sum(1 for c in expected_concepts if c in summary.lower())
summ_score = found / len(expected_concepts)
quality_scores['Summarization'] = summ_score
print(f'  Concept coverage: {found}/{len(expected_concepts)} → {summ_score:.1%}')
print(f'  Summary: {summary[:120]}')

overall_quality = sum(quality_scores.values()) / len(quality_scores)
print(f'\n=== Overall Quality Score: {overall_quality:.1%} ===')

In [ ]:
# Figure 9: Quality radar — real scores
categories = list(quality_scores.keys())
scores     = list(quality_scores.values())
cats_closed = categories + [categories[0]]
vals_closed = scores + [scores[0]]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=vals_closed, theta=cats_closed,
    fill='toself', name='Actual Scores',
    line_color='royalblue', opacity=0.7
))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Quality Benchmark Results (real model evaluation)',
    template='plotly_dark',
    height=520
)

save_plotly(fig, 'fig09_quality_radar')

## 13  Needle-in-Haystack Context Retrieval (real)

In [ ]:
import random
random.seed(42)

NEEDLE = 'The secret verification code is ALPHA-7742.'
FILLER = (
    'Researchers have been studying this phenomenon for decades. '
    'Technology continues to evolve at a rapid pace. '
    'Economic factors play a significant role in modern decisions. '
)

# Colab Pro can handle larger contexts — scale up to MAX_CONTEXT
niah_ctx_sizes = [c for c in [512, 1024, 2048, 4096, 8192, 16384, 32768] if c <= MAX_CONTEXT]
niah_depths    = [0.0, 0.25, 0.5, 0.75, 1.0]

niah_results = []

print('=== Needle-in-Haystack Benchmark ===')
print(f'Context sizes: {niah_ctx_sizes}, depths: {niah_depths}\n')

for ctx_len in niah_ctx_sizes:
    for depth in niah_depths:
        total_chars = ctx_len * 4
        needle_pos  = int((total_chars - len(NEEDLE)) * depth)
        prefix_len  = needle_pos
        suffix_len  = total_chars - needle_pos - len(NEEDLE)

        filler_long = (FILLER * ((total_chars // len(FILLER)) + 2))
        prefix = filler_long[:prefix_len]
        suffix = filler_long[prefix_len:prefix_len + suffix_len]

        document = prefix + NEEDLE + ' ' + suffix

        prompt = (
            f'Document:\n{document}\n\n'
            'Question: What is the secret verification code mentioned in the document? '
            'Answer with just the code:'
        )

        response = await model.generate(prompt, max_tokens=30, temperature=0)
        correct  = 'ALPHA-7742' in response or '7742' in response

        niah_results.append({
            'ctx_len': ctx_len, 'depth': depth,
            'correct': correct, 'response': response[:50]
        })

        status = '✓' if correct else '✗'
        print(f'  {status} ctx={ctx_len:6d}, depth={depth:.2f} → {response[:40]}')

overall_niah = sum(r['correct'] for r in niah_results) / len(niah_results)
print(f'\nOverall NIAH accuracy: {overall_niah:.1%}')

In [ ]:
# Figure 10: NIAH heatmap (real results)
import numpy as np

grid = np.zeros((len(niah_ctx_sizes), len(niah_depths)))
for r in niah_results:
    ri = niah_ctx_sizes.index(r['ctx_len'])
    ci = niah_depths.index(r['depth'])
    grid[ri, ci] = 1.0 if r['correct'] else 0.0

fig, ax = plt.subplots(figsize=(9, max(3, len(niah_ctx_sizes))))
im = ax.imshow(grid, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')

for ri in range(len(niah_ctx_sizes)):
    for ci in range(len(niah_depths)):
        ax.text(ci, ri, '✓' if grid[ri, ci] == 1 else '✗',
                ha='center', va='center', fontsize=14,
                color='black' if grid[ri, ci] > 0.5 else 'white')

ax.set_xticks(range(len(niah_depths)))
ax.set_xticklabels([f'{int(d*100)}%' for d in niah_depths])
ax.set_yticks(range(len(niah_ctx_sizes)))
ax.set_yticklabels([f'{c} tok' for c in niah_ctx_sizes])
ax.set_xlabel('Needle Depth (% into document)', fontsize=12)
ax.set_ylabel('Context Length', fontsize=12)
ax.set_title(f'Needle-in-Haystack Retrieval Accuracy (overall {overall_niah:.1%})',
             fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Correct')

save(fig, 'fig10_niah_heatmap')

## 14  Thread Scaling (real measurements at different n_threads)

In [ ]:
# Thread scaling is meaningful for CPU inference only.
# On GPU (n_gpu_layers=-1), the GPU handles all matrix ops and threads are irrelevant.
# We skip the expensive model-reload loop when GPU is in use and show a single measurement.

bench_prompt = 'Write a Python function to sort a list using merge sort with type hints.'
thread_tps = {}

if n_gpu_layers != 0:
    print('GPU inference active — thread scaling not applicable.')
    print('Measuring current throughput as the single GPU data point.\n')
    _ = await model.generate(bench_prompt, max_tokens=32)  # warmup
    samples = []
    for i in range(BENCH_ITERATIONS):
        t0  = time.perf_counter()
        out = await model.generate(bench_prompt, max_tokens=MAX_TOKENS)
        t1  = time.perf_counter()
        tps = int(len(out.split()) / 0.75) / (t1 - t0)
        samples.append(tps)
        print(f'  iter {i+1}: {tps:.2f} tok/s')
    thread_tps[config.n_threads] = _stat.mean(samples)
    print(f'\n  GPU throughput: {_stat.mean(samples):.2f} tok/s (all layers on GPU)')

else:
    max_threads  = psutil.cpu_count(logical=False) or 4
    thread_counts = sorted(set([1, 2, max(2, max_threads // 2), max_threads]))
    print(f'CPU inference — testing n_threads: {thread_counts}')
    print('(reloads model for each thread count)\n')

    for n_t in thread_counts:
        cfg = M7Config(
            n_threads=n_t,
            max_memory_gb=total_ram_gb * 0.85,
            n_ctx=MAX_CONTEXT,
        )
        _m = Miniforge.from_gguf(gguf_path, config=cfg)
        await _m.initialize()

        _ = await _m.generate(bench_prompt, max_tokens=32)
        samples = []
        for _ in range(3):
            t0  = time.perf_counter()
            out = await _m.generate(bench_prompt, max_tokens=MAX_TOKENS)
            t1  = time.perf_counter()
            tps = int(len(out.split()) / 0.75) / (t1 - t0)
            samples.append(tps)

        await _m.cleanup()
        mean_tps = _stat.mean(samples)
        thread_tps[n_t] = mean_tps
        print(f'  n_threads={n_t:2d}: {mean_tps:.2f} tok/s')

    # Reload original model
    await model.cleanup()
    model = Miniforge.from_gguf(gguf_path, config=config)
    await model.initialize()
    print('\nOriginal model reloaded.')

In [ ]:
# Figure 12: GPU throughput bar (single point) or CPU thread scaling (multiple)
fig, ax = plt.subplots(figsize=(9, 4))

t_vals   = sorted(thread_tps.keys())
tps_vals = [thread_tps[t] for t in t_vals]

if n_gpu_layers != 0:
    # Single GPU bar
    bars = ax.bar(['GPU (all layers)'], tps_vals, color='royalblue', width=0.4)
    for bar, y in zip(bars, tps_vals):
        ax.text(bar.get_x() + bar.get_width() / 2, y + max(tps_vals) * 0.02,
                f'{y:.1f} tok/s', ha='center', fontsize=12, fontweight='bold')
    ax.set_xlabel('Backend', fontsize=12)
    ax.set_title('GPU Inference Throughput — Miniforge / llama.cpp', fontsize=13, fontweight='bold')
else:
    ax.plot(t_vals, tps_vals, 'o-', color='steelblue', linewidth=2.5, markersize=9)
    ax.axvline(psutil.cpu_count(logical=False) or 4, color='green', linestyle='--',
               label=f'Physical cores ({psutil.cpu_count(logical=False) or 4})')
    for x, y in zip(t_vals, tps_vals):
        ax.annotate(f'{y:.1f}', (x, y), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=10)
    ax.set_xlabel('n_threads', fontsize=12)
    ax.set_xticks(t_vals)
    ax.legend()
    ax.set_title('CPU Thread Scaling — Miniforge / llama.cpp', fontsize=13, fontweight='bold')

ax.set_ylabel('Throughput (tok/s)', fontsize=12)

save(fig, 'fig12_throughput_scaling')

## 15  Auto-Quantization Heatmap (real MemoryManager)

In [ ]:
# Figure 5: Auto-selected quantization for different model sizes × RAM budgets
model_sizes = [1.3, 2.7, 7.0, 13.0, 30.0, 70.0]
budgets_gb  = [8, 12, 16, 20, 24, 32]

quant_order = ['Q2_K', 'Q3_K_M', 'Q4_K_M', 'Q5_K_M', 'Q6_K', 'Q8_0']
quant_to_idx = {q: i for i, q in enumerate(quant_order)}

grid  = np.zeros((len(model_sizes), len(budgets_gb)), dtype=float)
annot = np.empty_like(grid, dtype=object)

for r, params in enumerate(model_sizes):
    for c, budget in enumerate(budgets_gb):
        mgr = MemoryManager(target_utilization=budget / (budget + 4.0))
        q = mgr.select_quantization(params)
        grid[r, c]  = quant_to_idx.get(q, 0)
        annot[r, c] = q

fig, ax = plt.subplots(figsize=(10, 5))
cmap = plt.get_cmap('RdYlGn', len(quant_order))
im   = ax.imshow(grid, cmap=cmap, vmin=-0.5, vmax=len(quant_order) - 0.5, aspect='auto')

for r in range(len(model_sizes)):
    for c in range(len(budgets_gb)):
        ax.text(c, r, annot[r, c], ha='center', va='center',
                fontsize=9, color='black', fontweight='bold')

ax.set_xticks(range(len(budgets_gb)))
ax.set_xticklabels([f'{b} GB' for b in budgets_gb])
ax.set_yticks(range(len(model_sizes)))
ax.set_yticklabels([f'{p}B' for p in model_sizes])
ax.set_xlabel('Available RAM Budget', fontsize=12)
ax.set_ylabel('Model Parameters', fontsize=12)
ax.set_title('Auto-Selected Quantization (real MemoryManager output)', fontsize=14, fontweight='bold')

cbar = plt.colorbar(im, ax=ax, ticks=range(len(quant_order)))
cbar.ax.set_yticklabels(quant_order)
cbar.set_label('Quantization Quality →', rotation=270, labelpad=15)

save(fig, 'fig05_quantization_heatmap')

## 16  Optimal Settings Matrix (real MemoryManager)

In [ ]:
import pandas as pd

model_params = [1.3, 2.7, 7.0, 13.0]
all_settings = [mem.get_optimal_settings(p) for p in model_params]

df = pd.DataFrame(all_settings, index=[f'{p}B' for p in model_params])
cols = [c for c in ['quantization', 'n_ctx', 'cache_type_k', 'n_threads', 'n_batch', 'flash_attn']
        if c in df.columns]
df = df[cols]
df.index.name = 'Model'
print(df.to_string())

fig, ax = plt.subplots(figsize=(12, 3))
ax.axis('off')
tbl = ax.table(
    cellText=df.values, colLabels=df.columns, rowLabels=df.index,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.2, 2.0)

for (row, col), cell in tbl.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#37474f')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#eceff1')

ax.set_title('Auto-Derived Optimal Settings per Model Size (real MemoryManager)',
             fontsize=13, fontweight='bold', pad=20)

save(fig, 'fig11_optimal_settings_table')

## 17  Save Full JSON Report

In [ ]:
live_stats = mem.get_stats()

report = {
    'timestamp':          TIMESTAMP,
    'miniforge_version':  '0.1.0',
    'model': {
        'path':         str(gguf_path),
        'load_time_s':  round(load_time, 2),
        'n_threads':    config.n_threads,
        'n_ctx':        config.n_ctx,
        'n_gpu_layers': config.n_gpu_layers,
        'flash_attn':   config.flash_attn,
    },
    'hardware': {
        'cpu_physical_cores': psutil.cpu_count(logical=False),
        'cpu_logical_cores':  psutil.cpu_count(logical=True),
        'total_ram_gb':       round(live_stats.total_gb, 2),
        'available_ram_gb':   round(live_stats.available_gb, 2),
        'gpu_vram_mb':        vram_mb,
        'gpu_inference':      n_gpu_layers != 0,
    },
    'unit_tests': {
        'passed': passed, 'failed': failed,
        'errors': errors, 'skipped': skipped,
    },
    'throughput': {
        ptype: {
            'mean_tps': round(_stat.mean(vals), 2),
            'std_tps':  round(_stat.stdev(vals) if len(vals) > 1 else 0, 2),
            'samples':  [round(v, 2) for v in vals],
        }
        for ptype, vals in throughput_results.items()
    },
    'context_scaling': [
        {'ctx_len': c, 'ttft_ms': round(t, 1), 'tps': round(s, 2)}
        for c, t, s in zip(ctx_sizes, ctx_ttft, ctx_tps)
    ],
    'memory': {
        'peak_rss_mb': round(max(mem_snapshots), 1) if mem_snapshots else 0,
        'mean_rss_mb': round(float(np.mean(mem_snapshots)), 1) if mem_snapshots else 0,
    },
    'quality': {
        cat.replace('\n', ' '): round(score, 3)
        for cat, score in quality_scores.items()
    },
    'quality_overall': round(overall_quality, 3),
    'niah': {
        'overall_accuracy': round(overall_niah, 3),
        'context_sizes_tested': niah_ctx_sizes,
        'results': niah_results,
    },
    'throughput_scaling': {str(k): round(v, 2) for k, v in thread_tps.items()},
}

report_path = f'{OUTPUT_DIR}/miniforge_report_{TIMESTAMP}.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Report saved → {report_path}')
print(json.dumps({k: report[k] for k in
                  ['timestamp', 'model', 'hardware', 'quality_overall', 'niah']}, indent=2))

## 18  Cleanup

In [ ]:
await model.cleanup()
print('Model unloaded.')
print(f'\nAll figures and the JSON report are in: {OUTPUT_DIR}')